# Notebook 3 — Interactive Mapping with folium

**Goal:** go from Notebook 1's one-liner `.explore()` map to a fully custom,
multi-layer map — styled footprints, colored infrastructure points, a legend — built
directly with `folium`.

`/risk/features/multi.geojson` is the workhorse endpoint here: give it a dam number
and (optionally) which targets you want, or `all=true` for everything, and it
returns every intersecting feature — points, lines, and polygons — in a single
GeoJSON `FeatureCollection`, tagged with a `target` property so you know what each
feature is.


In [ ]:
import requests
import geopandas as gpd
import folium

BASE_URL = "http://149.165.154.170:30080"
DAM = "UT00221"  # Mountain Dell

features_json = requests.get(
    f"{BASE_URL}/risk/features/multi.geojson", params={"damnumber": DAM, "all": "true"}, timeout=20
).json()

features_gdf = gpd.GeoDataFrame.from_features(features_json["features"], crs="EPSG:4326")
features_gdf["target"].value_counts()


## Building the map layer by layer

We'll style three groups differently, since they need different treatment:
- **The inundation zone** — one polygon outline, no fill, so it doesn't hide everything else.
- **Point infrastructure** (hospitals, power plants, aviation, hazardous waste, wwtp) — colored circle markers with popups.
- **Lines and polygons** (railroads, transportation, gap_status, svi_tracts) — colored by type with a legend.


In [ ]:
zone_json = requests.get(f"{BASE_URL}/risk/zone.geojson", params={"damnumber": DAM}, timeout=10).json()
zone_gdf = gpd.GeoDataFrame.from_features(zone_json["features"], crs="EPSG:4326")

center = zone_gdf.union_all().centroid
m = folium.Map(location=[center.y, center.x], zoom_start=11, tiles="cartodbpositron")

folium.GeoJson(
    zone_gdf,
    name="Inundation zone",
    style_function=lambda f: {"color": "crimson", "weight": 2, "fillOpacity": 0},
).add_to(m)

m


In [ ]:
POINT_STYLE = {
    "hospitals": {"color": "red", "icon": "plus-sign"},
    "power_plants": {"color": "orange", "icon": "flash"},
    "aviation": {"color": "blue", "icon": "plane"},
    "hazardous_waste": {"color": "black", "icon": "warning-sign"},
    "wwtp": {"color": "darkgreen", "icon": "tint"},
    "dams": {"color": "purple", "icon": "home"},
}

point_targets = [t for t in POINT_STYLE if t in features_gdf["target"].unique()]
points_gdf = features_gdf[features_gdf["target"].isin(point_targets)]

for _, row in points_gdf.iterrows():
    style = POINT_STYLE[row["target"]]
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row["target"],
        icon=folium.Icon(color=style["color"], icon=style["icon"], prefix="glyphicon"),
    ).add_to(m)

m


In [ ]:
LINE_COLORS = {
    "railroads": "gray",
    "transportation": "steelblue",
    "ng_pipelines": "goldenrod",
    "transmission": "darkred",
}

for target, color in LINE_COLORS.items():
    subset = features_gdf[features_gdf["target"] == target]
    if subset.empty:
        continue
    folium.GeoJson(
        subset,
        name=target,
        style_function=lambda f, color=color: {"color": color, "weight": 2},
        tooltip=folium.GeoJsonTooltip(fields=["target"]),
    ).add_to(m)

m


In [ ]:
POLY_COLORS = {"gap_status": "forestgreen", "svi_tracts": "purple"}

for target, color in POLY_COLORS.items():
    subset = features_gdf[features_gdf["target"] == target]
    if subset.empty:
        continue
    folium.GeoJson(
        subset,
        name=target,
        style_function=lambda f, color=color: {"color": color, "fillColor": color, "weight": 1, "fillOpacity": 0.25},
        tooltip=folium.GeoJsonTooltip(fields=["target"]),
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m


## Exercise

Pick a different dam (try `"UT00755"`, Little Dell — it has the highest interstate
mileage impact in the dataset) and re-run this notebook top to bottom. Then try
adding a new layer for a target we didn't style above — `interstates_impact` or
`ushighway_impact` are good candidates (check `features_gdf["target"].unique()`
first to confirm they're present for your chosen dam).
